# Evaluation Summary
Produces figures and tables for the final report.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

results_dir = Path('../results')

In [ ]:
# Load evaluation CSVs
baseline = pd.read_csv(results_dir / 'eval_baseline.csv')
rag = pd.read_csv(results_dir / 'eval_rag.csv')

print('Baseline accuracy:', baseline['correct'].mean())
print('RAG accuracy:     ', rag['correct'].mean())

In [ ]:
# Bootstrap confidence intervals
def bootstrap_ci(correct_series, n_resamples=1000, ci=0.95):
    rng = np.random.default_rng(42)
    accs = []
    arr = correct_series.values
    for _ in range(n_resamples):
        sample = rng.choice(arr, size=len(arr), replace=True)
        accs.append(sample.mean())
    lo = np.percentile(accs, (1 - ci) / 2 * 100)
    hi = np.percentile(accs, (1 - (1 - ci) / 2) * 100)
    return lo, hi

b_lo, b_hi = bootstrap_ci(baseline['correct'])
r_lo, r_hi = bootstrap_ci(rag['correct'])
print(f'Baseline 95% CI: [{b_lo:.3f}, {b_hi:.3f}]')
print(f'RAG      95% CI: [{r_lo:.3f}, {r_hi:.3f}]')

In [ ]:
# Per-specialty accuracy
TARGET_SPECIALTIES = ['Surgery', 'Obstetrics & Gynecology', 'Pediatrics',
                      'Infectious Disease', 'Internal Medicine']

b_spec = baseline.groupby('specialty')['correct'].mean()
r_spec = rag.groupby('specialty')['correct'].mean()

spec_df = pd.DataFrame({'Baseline': b_spec, 'RAG': r_spec})
print(spec_df)

ax = spec_df.plot(kind='bar', figsize=(9, 4), rot=30)
ax.set_title('Per-specialty MCQ accuracy: Baseline vs RAG')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../report/figures/accuracy_by_specialty.png', dpi=150)
plt.show()

In [ ]:
# Dense vs sparse comparison
comp = pd.read_csv(results_dir / 'eval_retriever_comparison.csv')
print('Dense accuracy: ', comp['correct_dense'].mean())
print('Sparse accuracy:', comp['correct_sparse'].mean())

comp_spec = comp.groupby('specialty')[['correct_dense', 'correct_sparse']].mean()
print(comp_spec)

In [ ]:
# Latency and cost summary
print('=== Baseline ===')
print(baseline['latency_ms'].describe())
print('\n=== RAG ===')
print(rag['latency_ms'].describe())

# Estimated cost at current prices (claude-sonnet-4-6 Jan 2025: $3/$15 per 1M tokens)
PRICE_IN = 3.0 / 1e6
PRICE_OUT = 15.0 / 1e6
if 'input_tokens' in rag.columns:
    total_in = rag['input_tokens'].sum()
    total_out = rag['output_tokens'].sum()
    cost = total_in * PRICE_IN + total_out * PRICE_OUT
    print(f'\nEstimated cost for {len(rag)} RAG interactions: ${cost:.4f}')
    per_100 = cost / len(rag) * 100
    print(f'Cost per 100 interactions: ${per_100:.4f}')